# Tugas Akhir 06 (TA-06): Eksplorasi Mendalam Algoritma Random Forest
**Mata Praktikum:** Pembelajaran Mesin (Semester 5)  
**Dosen Pengampu:** Mohammad Bayu Anggara, S.Kom., M.Kom.  
**Dataset:** Breast Cancer Wisconsin (Diagnostic) Dataset (569 sampel, 30 fitur klinis)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Persiapan Data & Preprocessing
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')  # 0 = Malignant (Ganas), 1 = Benign (Jinak)

print('Ukuran Dataset:', X.shape)
print('Jumlah Missing Values:', X.isnull().sum().sum())
print('\nDistribusi Kelas Target:\n', y.value_counts())

# Split Data 80% Training, 20% Testing secara terstratifikasi
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# 2. Skenario A: Pengaruh Jumlah Pohon (n_estimators)
n_estimators_list = [10, 50, 100, 200, 500]
acc_train_A, acc_test_A = [], []

for n in n_estimators_list:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    rf.fit(X_train, y_train)
    acc_train_A.append(accuracy_score(y_train, rf.predict(X_train)))
    acc_test_A.append(accuracy_score(y_test, rf.predict(X_test)))

df_skenario_a = pd.DataFrame({
    'n_estimators': n_estimators_list,
    'Train Accuracy': acc_train_A,
    'Test Accuracy': acc_test_A
})
print(df_skenario_a)

plt.figure(figsize=(8, 4.5))
plt.plot(n_estimators_list, acc_test_A, marker='o', linewidth=2.5, label='Testing Accuracy')
plt.plot(n_estimators_list, acc_train_A, marker='s', linestyle='--', linewidth=2, label='Training Accuracy')
plt.title('Skenario A: Pengaruh Jumlah Pohon (n_estimators) terhadap Akurasi')
plt.xlabel('Jumlah Pohon (n_estimators)')
plt.ylabel('Akurasi')
plt.xticks(n_estimators_list)
plt.grid(True, linestyle=':')
plt.legend()
plt.show()

In [ ]:
# 3. Skenario B: Pengaruh Kedalaman Pohon (max_depth)
best_n = 50
max_depth_list = [None, 3, 5, 10, 20]
max_depth_labels = ['None (Unlimited)', '3', '5', '10', '20']
acc_train_B, acc_test_B = [], []

for d in max_depth_list:
    rf = RandomForestClassifier(n_estimators=best_n, max_depth=d, random_state=42)
    rf.fit(X_train, y_train)
    acc_train_B.append(accuracy_score(y_train, rf.predict(X_train)))
    acc_test_B.append(accuracy_score(y_test, rf.predict(X_test)))

df_skenario_b = pd.DataFrame({
    'max_depth': max_depth_labels,
    'Train Accuracy': acc_train_B,
    'Test Accuracy': acc_test_B
})
print(df_skenario_b)

plt.figure(figsize=(8, 4.5))
plt.plot(max_depth_labels, acc_train_B, marker='s', linewidth=2.5, label='Training Accuracy')
plt.plot(max_depth_labels, acc_test_B, marker='o', linewidth=2.5, label='Testing Accuracy')
plt.title(f'Skenario B: Pengaruh Kedalaman Pohon (max_depth) (n_estimators={best_n})')
plt.xlabel('Kedalaman Maksimum Pohon (max_depth)')
plt.ylabel('Akurasi')
plt.grid(True, linestyle=':')
plt.legend()
plt.show()

In [ ]:
# 4. Analisis Feature Importance & Evaluasi Model Terbaik
best_model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print('Akurasi Terbaik:', accuracy_score(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred, target_names=['Malignant (0)', 'Benign (1)']))

fi_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_model.feature_importances_
}).sort_values(by='Importance', ascending=False).head(10)

plt.figure(figsize=(8.5, 5))
sns.barplot(x='Importance', y='Feature', data=fi_df, palette='viridis')
plt.title('Top 10 Feature Importance - Random Forest Terbaik')
plt.xlabel('Tingkat Kepentingan')
plt.ylabel('Fitur')
plt.show()

# Simpan model untuk deployment web
joblib.dump(best_model, 'rf_breast_cancer_model.pkl')